## Environment

```{bash}
pip install "decoupler>=1.4.0"

```

### Import generic packages

In [ ]:
import scanpy as sc
import decoupler as dc
import plotnine as p9
import liana as li
import numpy as np

## Load and Normalize Data

We will use an ischemic 10X Visium spatial slide from [Kuppe et al., 2022](https://www.nature.com/articles/s41586-022-05060-x). It is a tissue sample obtained from a patient with myocardial infarction, specifically focusing on the ischemic zone of the heart tissue. 

The slide provides spatially-resolved information about the cellular composition and gene expression patterns within the tissue.

In [ ]:
adata = sc.read("kuppe_heart19.h5ad", backup_url='https://figshare.com/ndownloader/files/41501073?private_link=4744950f8768d5c8f68c')

In [ ]:
adata.obs.head()

Normalize data

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

Spot clusters

In [ ]:
sc.pl.spatial(adata, color=[None, 'celltype_niche'], size=1.3, palette='Set1')

##### Extract Cell type Composition
This slide comes with estimated cell type proportions using cell2location; See [Kuppe et al., 2022](https://www.nature.com/articles/s41586-022-05060-x). Let's extract from .obsm them to an independent AnnData object.

In [ ]:
# Rename to more informative names
full_names = {'Adipo': 'Adipocytes',
              'CM': 'Cardiomyocytes',
              'Endo': 'Endothelial',
              'Fib': 'Fibroblasts',
              'PC': 'Pericytes',
              'prolif': 'Proliferating',
              'vSMCs': 'Vascular_SMCs',
              }
# but only for the ones that are in the data
adata.obsm['compositions'].columns = [full_names.get(c, c) for c in adata.obsm['compositions'].columns]

In [ ]:
li.ut.spatial_neighbors(adata, bandwidth=200, cutoff=0.1, kernel='gaussian', set_diag=True)

In [ ]:
from liana.mt import inflow

In [ ]:
lrdata = inflow(adata=adata, 
                resource_name='consensus', 
                obsm_key='compositions',
                use_raw=False,
                nz_prop=0.05
                )

In [ ]:
lrdata.var

In [ ]:
lrdata.var['var_mean'] = lrdata.X.mean(axis=0).A1

In [ ]:
lrdata.var.sort_values('var_mean').tail(20)

In [ ]:
sc.pl.spatial(lrdata, color=['Fibroblasts^COL1A1^CD36', 'Fibroblasts^FN1^ITGAV_ITGB1'], size=1.3, cmap='viridis')

## Test on Multimodal Data

In [ ]:
import mudata as mu

In [ ]:
# let's download the data
rna = sc.read("sma_rna.h5ad", backup_url="https://figshare.com/ndownloader/files/44624974?private_link=4744950f8768d5c8f68c")
msi = sc.read("sma_msi.h5ad", backup_url="https://figshare.com/ndownloader/files/44624971?private_link=4744950f8768d5c8f68c")
ct = sc.read("sma_ct.h5ad", backup_url="https://figshare.com/ndownloader/files/44624968?private_link=4744950f8768d5c8f68c")

In [ ]:
rna.X.max() # log transformed

In [ ]:
msi.X.max() # total count transformed

In [ ]:
# Required to bring msi to the same scale as rna
msi = li.ut.interpolate_adata(target=msi, reference=rna, use_raw=False, spatial_key='spatial')

In [ ]:
# and create a MuData object
mdata = mu.MuData({'rna':rna, 'msi':msi, 'ct':ct})
mdata

In [ ]:
# copy spatial coords & compute neighbors
mdata.uns = rna.uns
mdata.obsm = rna.obsm
li.ut.spatial_neighbors(mdata, bandwidth=200, cutoff=0.1, kernel='gaussian', set_diag=True)

In [ ]:
# convert ct to dataframe and add to mdata
mdata.obsm['comp'] = ct.to_df()

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, format='png', frameon=False, transparent=True, figsize=[5,5])

In [ ]:
# set global figure parameters
kwargs = {'frameon':False, 'size':1.5, 'img_key':'lowres'}

In [ ]:
sc.pl.spatial(msi, color='Dopamine', cmap='magma', **kwargs)

In [ ]:
metalinks = li.rs.get_metalinks(tissue_location='Brain',
                                biospecimen_location='Cerebrospinal Fluid (CSF)',
                                source=['CellPhoneDB', 'NeuronChat']
                                )
metalinks.head()

In [ ]:
map_df = li.rs.get_hcop_orthologs(columns=['human_symbol', 'mouse_symbol'],
                                  min_evidence=3
                                  ).rename(columns={'human_symbol':'source',
                                                   'mouse_symbol':'target'})

In [ ]:
metalinks = li.rs.translate_column(resource=metalinks,
                                   map_df=map_df,
                                   column='gene_symbol',
                                   one_to_many=1)
metalinks.head()

In [ ]:
interactions = metalinks[['metabolite', 'gene_symbol']].apply(tuple, axis=1).tolist()

In [ ]:
lrdata = li.mt.inflow(adata=mdata, 
                      x_mod='msi',
                      y_mod='rna',
                      obsm_key='comp',
                      interactions=interactions,
                      x_use_raw=False,
                      y_use_raw=False,
                      x_transform=li.ut.zi_minmax,
                      x_transform_kwargs={'cutoff':0.001},
                      y_transform=li.ut.zi_minmax,
                      y_transform_kwargs={'cutoff':0.01}
                      )

In [ ]:
# some quick summaries
lrdata.var['mean'] = lrdata.X.mean(axis=0).A1
lrdata.var['std'] = lrdata.X.A.std(axis=0)
lrdata.var['cv'] = lrdata.var['std'] / (lrdata.var['mean'] + 1e-6)

In [ ]:
lrdata.var.sort_values("cv", ascending=False, key=abs).head(10)

In [ ]:
sc.pl.spatial(lrdata, color=["MSN1^Dopamine^Drd1", "Peptidergic neurons^Serotonin^Htr1f"], spot_size=500)

# TODO: we really need to filter the interactions better...

In [ ]:
sc.pl.spatial(adata=rna, color='Drd1')

In [ ]:
sc.pl.spatial(adata=msi, color='Dopamine')

In [ ]:
sc.pl.spatial(adata=ct, color='MSN1')